# Treino Simples de Machine Learning com Dados de CGM (3 Pacientes)

Notebook enxuto: pega 3 sujeitos do dataset, monta um conjunto de treino com a última 1h de
leituras de glicose e treina uma **Regressão Linear** para prever a próxima leitura.


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib

## 1. Conectar ao Google Drive e localizar o dataset

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SEARCH_ROOT = Path("/content/drive/MyDrive")
except ImportError:
    SEARCH_ROOT = Path.cwd()

def find_dataset_dir(search_root, folder_name="CGM Records"):
    for cand in search_root.rglob(folder_name):
        if cand.is_dir() and list(cand.glob("Subject */Subject *.csv")):
            return cand
    return None

DATASET_DIR = find_dataset_dir(SEARCH_ROOT)
if DATASET_DIR is None:
    raise FileNotFoundError("Pasta 'CGM Records' não encontrada no Drive.")

print("Dataset encontrado em:", DATASET_DIR)

## 2. Configuração

In [ ]:
SUBJECTS = ["Subject 1", "Subject 2", "Subject 3"]
WINDOW = 12          # 12 leituras de 5 min = 1h de histórico
TRAIN_FRAC = 0.8

## 3. Carregar os dados de cada paciente (CGM, insulina e carboidratos)

Além do CGM, o CSV de cada sujeito já traz insulina (`CorrectionDelivered` +
`TotalBolusInsulinDelivered`) e carboidratos (`CarbSize` + `FoodDelivered`). Esses dois eventos
são bem esparsos (a maior parte do tempo é zero, só tem valor no instante da dose/refeição), então
em vez de usar o valor bruto a cada 5 min, calculamos a **soma acumulada** em uma janela recente:

- `insulina_acumulada`: soma da insulina aplicada nas últimas **4h** (tempo aproximado de ação da
  insulina no corpo);
- `carboidratos_acumulados`: soma dos carboidratos consumidos nas últimas **2h** (janela em que eles
  mais afetam a glicose).

Isso captura o efeito residual desses eventos mesmo quando não há uma dose/refeição exatamente na
hora prevista.


In [ ]:
import pandas as pd

INSULIN_HORAS = 4       # duração aproximada da ação da insulina
CARB_HORAS = 2          # janela de maior efeito dos carboidratos na glicose

def load_subject(subject_name):
    path = DATASET_DIR / subject_name / f"{subject_name}.csv"
    df = pd.read_csv(path)
    df["EventDateTime"] = pd.to_datetime(df["EventDateTime"])
    df = df.sort_values("EventDateTime").set_index("EventDateTime")

    agg = df.resample("5min").agg({
        "CGM": "mean",
        "CorrectionDelivered": "sum",
        "TotalBolusInsulinDelivered": "sum",
        "CarbSize": "sum",
        "FoodDelivered": "sum",
    })
    agg["CGM"] = agg["CGM"].interpolate(limit=6)

    doses = ["CorrectionDelivered", "TotalBolusInsulinDelivered", "CarbSize", "FoodDelivered"]
    agg[doses] = agg[doses].fillna(0.0)

    agg["insulina"] = agg["CorrectionDelivered"] + agg["TotalBolusInsulinDelivered"]
    agg["carboidratos"] = agg["CarbSize"] + agg["FoodDelivered"]

    agg["insulina_acumulada"] = agg["insulina"].rolling(INSULIN_HORAS * 12, min_periods=1).sum()
    agg["carboidratos_acumulados"] = agg["carboidratos"].rolling(CARB_HORAS * 12, min_periods=1).sum()

    agg = agg.dropna(subset=["CGM"])
    return agg

## 4. Montar janelas de treino

Cada amostra vira: **[últimas 12 leituras de CGM] + [insulina acumulada] + [carboidratos
acumulados]** para prever a próxima leitura de glicose. São 2 parâmetros novos, além da janela de
CGM que já tínhamos.


In [ ]:
import numpy as np

def build_windows(df, window=WINDOW):
    cgm = df["CGM"].values
    insulina = df["insulina_acumulada"].values
    carboidratos = df["carboidratos_acumulados"].values

    X, y = [], []
    for i in range(window, len(df)):
        janela_cgm = cgm[i - window:i]
        features = np.concatenate([janela_cgm, [insulina[i - 1]], [carboidratos[i - 1]]])
        X.append(features)
        y.append(cgm[i])
    return np.array(X), np.array(y)

## 5. Juntar os 3 pacientes em treino e teste

Cada paciente é dividido cronologicamente (80% treino / 20% teste) antes de juntar todos.


In [ ]:
X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

for subject in SUBJECTS:
    df = load_subject(subject)
    n_train = int(len(df) * TRAIN_FRAC)
    df_train, df_test = df.iloc[:n_train], df.iloc[n_train:]

    Xtr, ytr = build_windows(df_train)
    Xte, yte = build_windows(df_test)

    X_train_list.append(Xtr); y_train_list.append(ytr)
    X_test_list.append(Xte); y_test_list.append(yte)

    print(f"{subject}: {len(Xtr)} janelas de treino | {len(Xte)} de teste")

X_train = np.concatenate(X_train_list)
y_train = np.concatenate(y_train_list)
X_test = np.concatenate(X_test_list)
y_test = np.concatenate(y_test_list)

## 6. Treinar o modelo (Regressão Linear)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()

model = LinearRegression()
model.fit(X_train_s, y_train_s)

## 7. Avaliar no conjunto de teste

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

pred_s = model.predict(X_test_s)
pred = scaler_y.inverse_transform(pred_s.reshape(-1, 1)).ravel()

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print(f"MAE:  {mae:.2f} mg/dL")
print(f"RMSE: {rmse:.2f} mg/dL")

## 7.05 O que o modelo aprendeu sobre insulina e carboidratos?

Como o modelo é uma Regressão Linear, dá pra olhar diretamente o peso (coeficiente) que ele deu
para cada entrada. O sinal do coeficiente já diz se faz sentido fisiologicamente: insulina deveria
**baixar** a glicose prevista (coeficiente negativo) e carboidratos deveriam **subir** (coeficiente
positivo).


In [ ]:
coef_insulina, coef_carboidratos = model.coef_[-2], model.coef_[-1]
print(f"Coeficiente da insulina acumulada:       {coef_insulina:+.4f}")
print(f"Coeficiente dos carboidratos acumulados: {coef_carboidratos:+.4f}")
print(f"Coeficiente médio de 1 leitura de CGM:   {np.mean(np.abs(model.coef_[:-2])):.4f}")

**Importante ser honesto sobre o resultado aqui:** como este notebook prevê só a **próxima
leitura (5 min à frente)**, a própria janela de CGM já explica quase toda a variação — a glicose
não muda muito em 5 minutos, então o modelo aprende basicamente "o próximo valor será parecido com
o atual", e sobra pouco espaço para insulina/carboidratos melhorarem a previsão (o MAE quase não
muda). Os coeficientes devem sair pequenos, mas com o **sinal fisiológico correto** — isso já é um
bom indício de que as variáveis fazem sentido, mesmo contribuindo pouco no MAE final.

Esses dois parâmetros tendem a importar muito mais se o horizonte de previsão for maior (ex.: 30
min, como no enunciado original do trabalho) ou se for usado um modelo não linear (Random Forest,
Gradient Boosting, LSTM), que consegue capturar melhor o efeito atrasado da insulina/carboidratos
na glicose.


## 7.1 Accuracy, F1 e outras métricas de classificação

Prever glicose é um problema de **regressão** (o valor é um número contínuo, tipo 142 mg/dL), então
`accuracy` e `f1` não se aplicam direto — essas métricas são para quando a saída é uma **categoria**.

Um jeito comum de usar essas métricas aqui é converter os valores de glicose em **faixas clínicas**
(hipoglicemia / normal / hiperglicemia) e comparar a faixa prevista com a faixa real, como se fosse
um problema de classificação:

- Hipoglicemia: CGM < 70 mg/dL
- Normal: 70–180 mg/dL
- Hiperglicemia: CGM > 180 mg/dL


In [ ]:
from sklearn.metrics import (
    r2_score, accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

# R² também é comum reportar junto com MAE/RMSE em regressão
r2 = r2_score(y_test, pred)
print(f"R²: {r2:.4f}\n")

def categorizar(valores):
    bins = [-np.inf, 70, 180, np.inf]
    rotulos = ["Hipoglicemia", "Normal", "Hiperglicemia"]
    return pd.cut(valores, bins=bins, labels=rotulos)

y_test_cat = categorizar(y_test)
pred_cat = categorizar(pred)

print("Accuracy:            %.4f" % accuracy_score(y_test_cat, pred_cat))
print("Precision (macro):   %.4f" % precision_score(y_test_cat, pred_cat, average="macro", zero_division=0))
print("Recall (macro):      %.4f" % recall_score(y_test_cat, pred_cat, average="macro", zero_division=0))
print("F1-score (macro):    %.4f" % f1_score(y_test_cat, pred_cat, average="macro", zero_division=0))
print("F1-score (weighted): %.4f" % f1_score(y_test_cat, pred_cat, average="weighted", zero_division=0))

print("\nRelatório de classificação:")
print(classification_report(y_test_cat, pred_cat, zero_division=0))

print("Matriz de confusão:")
rotulos = ["Hipoglicemia", "Normal", "Hiperglicemia"]
cm = confusion_matrix(y_test_cat, pred_cat, labels=rotulos)
print(pd.DataFrame(cm, index=rotulos, columns=rotulos))

## 8. Visualizar previsto vs. real

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(y_test[:300], label="CGM real")
plt.plot(pred[:300], label="CGM previsto", alpha=0.8)
plt.xlabel("Amostras de teste (ordem no tempo)")
plt.ylabel("Glicose (mg/dL)")
plt.legend()
plt.tight_layout()
plt.show()